In [ ]:
# Cell 0 (FINAL): One-time disk patcher — run ONCE then RESTART RUNTIME.

import pathlib, site

SP = pathlib.Path(site.getsitepackages()[0])
print(f"site-packages: {SP}")

# ── Patch 1: transformers/pytorch_utils.py ───────────────────────────────────
pt_file = SP / "transformers" / "pytorch_utils.py"
src = pt_file.read_text(encoding="utf-8")
PATCH_TRANSFORMERS = '''
# >>> PATCHED: isin_mps_friendly for coqui-tts compat <
import torch as _torch
def isin_mps_friendly(elements, test_elements):
    if elements.dtype != test_elements.dtype:
        test_elements = test_elements.to(dtype=elements.dtype)
    return _torch.isin(elements, test_elements)
# >>> END PATCH <
'''
if "isin_mps_friendly" not in src:
    pt_file.write_text(src + PATCH_TRANSFORMERS, encoding="utf-8")
    print("✅ Patched transformers/pytorch_utils.py")
else:
    print("ℹ️  transformers/pytorch_utils.py already patched")

# ── Patch 2: datasets/features/_torchcodec.py ────────────────────────────────
tc_file = SP / "datasets" / "features" / "_torchcodec.py"
PATCH_TORCHCODEC = '''# patched by Colab setup — soundfile-backed AudioDecoder
import io
import numpy as np
import soundfile as _sf


class _Metadata:
    def __init__(self, sample_rate=16000):
        self.path         = None
        self.sample_rate  = sample_rate
        self.num_frames   = 0
        self.num_channels = 1
        self.bits_per_sample = 16
        self.encoding     = "PCM_S"


class _SampleList:
    def __init__(self, array, sample_rate):
        self.data        = array
        self.sample_rate = sample_rate


class AudioDecoder:
    def __init__(self, source, stream_index=0, sample_rate=None, **kwargs):
        if isinstance(source, (bytes, bytearray, memoryview)):
            buf = io.BytesIO(bytes(source))
        elif isinstance(source, io.IOBase):
            buf = source
        elif source is None:
            buf = None
        else:
            buf = str(source)
        if buf is not None:
            try:
                data, sr = _sf.read(buf, dtype="float32", always_2d=False)
            except Exception:
                data, sr = np.zeros(1600, dtype=np.float32), 16000
        else:
            data, sr = np.zeros(1600, dtype=np.float32), 16000
        if data.ndim > 1:
            data = data.mean(axis=1)
        if sample_rate is not None and sample_rate != sr:
            try:
                import scipy.signal as _sig
                n    = int(len(data) * sample_rate / sr)
                data = _sig.resample(data, n).astype(np.float32)
                sr   = sample_rate
            except Exception:
                pass
        self._samples    = _SampleList(data, sr)
        self._hf_encoded = {}
        self.metadata    = _Metadata(sample_rate=sr)
        self.metadata.num_frames   = len(data)
        self.metadata.num_channels = 1

    def get_all_samples(self):
        return self._samples

_AudioDecoder = AudioDecoder
'''
tc_file.write_text(PATCH_TORCHCODEC, encoding="utf-8")
print("✅ Patched datasets/features/_torchcodec.py")

# ── Patch 3: huggingface_hub/utils/_auth.py ──────────────────────────────────
hf_auth  = SP / "huggingface_hub" / "utils" / "_auth.py"
auth_src = hf_auth.read_text(encoding="utf-8")
PATCH_AUTH = '''
# >>> PATCHED: _save_stored_tokens stub <
if "_save_stored_tokens" not in dir():
    def _save_stored_tokens(tokens: dict) -> None:
        pass
# >>> END PATCH <
'''
if "_save_stored_tokens" not in auth_src:
    hf_auth.write_text(auth_src + PATCH_AUTH, encoding="utf-8")
    print("✅ Patched huggingface_hub/utils/_auth.py")
else:
    print("ℹ️  huggingface_hub/utils/_auth.py already patched")

# ── Patch 4: TTS/tts/models/xtts.py — replace load_audio with soundfile ──────
# load_audio() calls torchaudio.load() which collides with our AudioDecoder stub.
# We replace the whole function body to use soundfile + torch directly.
xtts_file = SP / "TTS" / "tts" / "models" / "xtts.py"
xtts_src  = xtts_file.read_text(encoding="utf-8")

OLD_LOAD_AUDIO = '''def load_audio(audiopath, sampling_rate):
    # better load since it checks for phoneme singletons
    x, sr = torchaudio.load(audiopath)
    if sr != sampling_rate:
        x = torchaudio.functional.resample(x, sr, sampling_rate)
    if x.size(0) != 1:
        x = x[0].unsqueeze(0)
    return x'''

NEW_LOAD_AUDIO = '''def load_audio(audiopath, sampling_rate):
    # patched: use soundfile instead of torchaudio to avoid AudioDecoder stub clash
    import soundfile as _sf
    import numpy as _np
    import scipy.signal as _sig
    data, sr = _sf.read(str(audiopath), dtype="float32", always_2d=False)
    if data.ndim > 1:
        data = data.mean(axis=1)
    if sr != sampling_rate:
        n    = int(len(data) * sampling_rate / sr)
        data = _sig.resample(data, n).astype(_np.float32)
    # Return as (1, T) torch tensor — same shape torchaudio would return
    import torch as _torch
    return _torch.from_numpy(data).unsqueeze(0)'''

if "patched: use soundfile" not in xtts_src:
    if OLD_LOAD_AUDIO in xtts_src:
        xtts_src = xtts_src.replace(OLD_LOAD_AUDIO, NEW_LOAD_AUDIO)
        xtts_file.write_text(xtts_src, encoding="utf-8")
        print("✅ Patched TTS/tts/models/xtts.py — load_audio now uses soundfile")
    else:
        # Fallback: find and replace via line-level search
        lines     = xtts_src.splitlines()
        new_lines = []
        skip      = False
        inserted  = False
        for i, line in enumerate(lines):
            if "def load_audio(audiopath, sampling_rate):" in line and not inserted:
                # Replace next few lines until we clear the function body
                new_lines.append(line)
                new_lines.append("    # patched: use soundfile instead of torchaudio")
                new_lines.append("    import soundfile as _sf")
                new_lines.append("    import numpy as _np")
                new_lines.append("    import scipy.signal as _sig")
                new_lines.append("    data, sr = _sf.read(str(audiopath), dtype='float32', always_2d=False)")
                new_lines.append("    if data.ndim > 1:")
                new_lines.append("        data = data.mean(axis=1)")
                new_lines.append("    if sr != sampling_rate:")
                new_lines.append("        n = int(len(data) * sampling_rate / sr)")
                new_lines.append("        data = _sig.resample(data, n).astype(_np.float32)")
                new_lines.append("    import torch as _torch")
                new_lines.append("    return _torch.from_numpy(data).unsqueeze(0)")
                skip    = True
                inserted = True
                continue
            if skip:
                # skip original function body lines until next top-level def/class
                stripped = line.rstrip()
                if stripped and not stripped.startswith("    ") and not stripped.startswith("#"):
                    skip = False
                    new_lines.append(line)
                # else skip the old body line
                continue
            new_lines.append(line)
        xtts_file.write_text("\n".join(new_lines), encoding="utf-8")
        print("✅ Patched TTS/tts/models/xtts.py (fallback line-level patch)")
else:
    print("ℹ️  TTS/tts/models/xtts.py already patched")

print("\n" + "="*60)
print("ALL PATCHES WRITTEN. → Runtime → Restart session")
print("After restart: skip Cell 0, run Cell 1 onward.")
print("="*60)

site-packages: /usr/local/lib/python3.12/dist-packages
ℹ️  transformers/pytorch_utils.py already patched
✅ Patched datasets/features/_torchcodec.py
ℹ️  huggingface_hub/utils/_auth.py already patched
✅ Patched TTS/tts/models/xtts.py (fallback line-level patch)

ALL PATCHES WRITTEN. → Runtime → Restart session
After restart: skip Cell 0, run Cell 1 onward.


In [ ]:
import torch, soundfile as _sf4, numpy as _np4, scipy.signal as _sig4, traceback
import tempfile, soundfile as sf_t, numpy as np_t, scipy.signal as sig_t

# ── Define the replacement ────────────────────────────────────────────────────
def _load_audio_final(audiopath, sampling_rate):
    data, sr = _sf4.read(str(audiopath), dtype="float32", always_2d=False)
    if data.ndim > 1:
        data = data.mean(axis=1)
    if sr != sampling_rate:
        n    = int(len(data) * sampling_rate / sr)
        data = _sig4.resample(data, n).astype(_np4.float32)
    data = _np4.clip(data, -1.0, 1.0)
    return torch.from_numpy(data).unsqueeze(0)

# ── Find every method on the model that references load_audio ────────────────
# and patch ALL of their globals dicts
import types

patched_globals = set()
methods_patched = []

for attr_name in dir(model):
    try:
        attr = getattr(model, attr_name)
    except Exception:
        continue
    # bound method
    if isinstance(attr, types.MethodType):
        fn = attr.__func__
        g  = fn.__globals__
        gid = id(g)
        if gid not in patched_globals:
            # check if this globals dict has a load_audio key or if xtts.py is the source
            src_file = g.get("__file__", "") or g.get("__spec__", None) and ""
            if "load_audio" in g or "xtts" in str(g.get("__file__", "")):
                g["load_audio"] = _load_audio_final
                patched_globals.add(gid)
                methods_patched.append(attr_name)

print(f"✅ Patched {len(patched_globals)} unique globals dicts")
print(f"   Methods covered: {methods_patched[:10]}")

# ── Also brute-force: patch the exact globals of get_conditioning_latents ─────
gcl_globals = model.get_conditioning_latents.__func__.__globals__
gcl_globals["load_audio"] = _load_audio_final
print(f"✅ Force-patched get_conditioning_latents globals (id={id(gcl_globals)})")
print(f"   load_audio now: {gcl_globals['load_audio']}")

# ── Also patch _clone_voice globals ──────────────────────────────────────────
cv_globals = model._clone_voice.__func__.__globals__
cv_globals["load_audio"] = _load_audio_final
print(f"✅ Force-patched _clone_voice globals (id={id(cv_globals)})")

# ── Confirm all three point to same function ──────────────────────────────────
print(f"\nget_conditioning_latents → load_audio: {gcl_globals.get('load_audio')}")
print(f"_clone_voice             → load_audio: {cv_globals.get('load_audio')}")

# ── Test ──────────────────────────────────────────────────────────────────────
data_t, sr_t = sf_t.read("/content/xtts_hindi_eval/reference/male_reference.wav",
                          dtype="float32", always_2d=False)
if data_t.ndim > 1: data_t = data_t.mean(axis=1)
if sr_t != 22050:
    data_t = sig_t.resample(data_t, int(len(data_t)*22050/sr_t)).astype("float32")
tmp_t = tempfile.NamedTemporaryFile(suffix=".wav", delete=False)
sf_t.write(tmp_t.name, data_t, 22050)

try:
    out = model.synthesize(
        "एक आम आदमी और औरत इमली के पेड़ के नीचे बैठकर ऊन बुन रहे हैं।",
        config, speaker_wav=tmp_t.name, language="hi",
        temperature=0.65, repetition_penalty=5.0,
        top_k=50, top_p=0.85, speed=1.0, enable_text_splitting=True,
    )
    wav_t = np_t.array(out["wav"], dtype=np_t.float32)
    sf_t.write("/content/test_hi_final.wav", wav_t, 24000)
    print(f"\n✅ SUCCESS — shape: {wav_t.shape}, saved to /content/test_hi_final.wav")
except Exception:
    traceback.print_exc()

✅ Patched 2 unique globals dicts
   Methods covered: ['__init__', 'eval_log']
✅ Force-patched get_conditioning_latents globals (id=137988838095104)
   load_audio now: <function _load_audio_final at 0x7d7fe3167240>
✅ Force-patched _clone_voice globals (id=137988806753728)

get_conditioning_latents → load_audio: <function _load_audio_final at 0x7d7fe3167240>
_clone_voice             → load_audio: <function _load_audio_final at 0x7d7fe3167240>

✅ SUCCESS — shape: (125696,), saved to /content/test_hi_final.wav


In [ ]:
# Cell 1 (FINAL): Install coqui-tts without fighting the HF stack.
# Strategy: let pip resolve dependencies freely, then fix broken imports
# via monkey-patches in Cell 2. No version pins = no resolution conflicts.

import subprocess, sys

def run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print("STDERR:", r.stderr[-2000:])
        raise RuntimeError(f"Command failed: {' '.join(cmd)}")
    return r

# Step 1: Wipe any partial installs from previous attempts
print("🧹 Removing any partial previous installs…")
run([sys.executable, "-m", "pip", "uninstall", "-y",
     "TTS", "coqui-tts", "trainer", "coqpit"])
run([sys.executable, "-m", "pip", "cache", "purge"])

# Step 2: Install coqui-tts — let pip pick compatible HF versions freely
print("📦 Installing coqui-tts…")
run([sys.executable, "-m", "pip", "install", "-q", "coqui-tts"])

# Step 3: Audio + data utilities
print("📦 Installing audio utilities…")
run([sys.executable, "-m", "pip", "install", "-q", "soundfile", "librosa", "scipy"])

# Step 4: Report installed versions of the HF stack (for debugging)
import importlib, subprocess
for pkg in ["coqui_tts", "transformers", "huggingface_hub", "datasets",
            "soundfile", "librosa"]:
    try:
        mod = importlib.import_module(pkg.replace("-", "_").split(".")[0])
        ver = getattr(mod, "__version__", "?")
        print(f"  {pkg}: {ver}")
    except Exception:
        # Some packages report version differently
        r = subprocess.run([sys.executable, "-m", "pip", "show", pkg.replace("_", "-")],
                           capture_output=True, text=True)
        for line in r.stdout.splitlines():
            if line.startswith("Version:"):
                print(f"  {pkg}: {line.split()[-1]}")
                break

print("\n✅ Cell 1 done. Now run Cell 2 (patches) before anything else.")
print("⚠️  If you see import errors above, do Runtime → Restart session, then skip Cell 1 and run Cell 2 directly.")

🧹 Removing any partial previous installs…
📦 Installing coqui-tts…
📦 Installing audio utilities…
  coqui_tts: 0.27.5
  transformers: 5.12.1
  huggingface_hub: 1.20.1
  datasets: 4.0.0
  soundfile: 0.14.0
  librosa: 0.11.0

✅ Cell 1 done. Now run Cell 2 (patches) before anything else.
⚠️  If you see import errors above, do Runtime → Restart session, then skip Cell 1 and run Cell 2 directly.


In [ ]:
# Cell 2 (POST-RESTART): Verify disk patches loaded cleanly + torchcodec blocker
# No reloads, no module cache manipulation — patches are already on disk.

import sys, types, importlib.abc, importlib.machinery, torch

# ── Verify patch 1: isin_mps_friendly ────────────────────────────────────────
from transformers.pytorch_utils import isin_mps_friendly
print("✅ isin_mps_friendly importable")

# ── Verify patch 2: datasets AudioDecoder ────────────────────────────────────
from datasets.features._torchcodec import AudioDecoder
print(f"✅ datasets AudioDecoder: {AudioDecoder}")

# ── Verify patch 3: huggingface_hub login ────────────────────────────────────
from huggingface_hub import login
print("✅ huggingface_hub login importable")

# ── torchcodec MetaPathFinder (TTS still needs this) ─────────────────────────
_TORCHCODEC_NAMES = {
    "torchcodec",
    "torchcodec.decoders",
    "torchcodec.decoders._video_decoder",
}

class _TorchcodecBlocker(importlib.abc.MetaPathFinder, importlib.abc.Loader):
    def find_spec(self, fullname, path, target=None):
        if fullname in _TORCHCODEC_NAMES:
            return importlib.machinery.ModuleSpec(
                name=fullname, loader=self,
                is_package=(fullname in {"torchcodec", "torchcodec.decoders"}),
            )
        return None
    def create_module(self, spec):
        if spec.name in sys.modules:
            return sys.modules[spec.name]
        mod = types.ModuleType(spec.name)
        mod.__spec__    = spec
        mod.__loader__  = self
        mod.__package__ = spec.parent
        mod.__path__    = [] if spec.submodule_search_locations is not None else None
        mod.__file__    = None
        return mod
    def exec_module(self, module):
        sys.modules[module.__spec__.name] = module

sys.meta_path = [f for f in sys.meta_path
                 if not isinstance(f, _TorchcodecBlocker)]
sys.meta_path.insert(0, _TorchcodecBlocker())

import importlib as _il
for _name in ["torchcodec", "torchcodec.decoders", "torchcodec.decoders._video_decoder"]:
    sys.modules.pop(_name, None)
    _il.import_module(_name)

sys.modules["torchcodec"].decoders = sys.modules["torchcodec.decoders"]
sys.modules["torchcodec.decoders"]._video_decoder = \
    sys.modules["torchcodec.decoders._video_decoder"]
# Give torchcodec.decoders the same working AudioDecoder that datasets uses
sys.modules["torchcodec.decoders"].AudioDecoder = AudioDecoder
print("✅ torchcodec MetaPathFinder installed + AudioDecoder wired")

# ── Final TTS import check ────────────────────────────────────────────────────
from TTS.api import TTS
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import Xtts
print("✅ TTS, XttsConfig, Xtts imported — proceed to Cell 3")

✅ isin_mps_friendly importable
✅ datasets AudioDecoder: <class 'datasets.features._torchcodec.AudioDecoder'>
✅ huggingface_hub login importable
✅ torchcodec MetaPathFinder installed + AudioDecoder wired
✅ TTS, XttsConfig, Xtts imported — proceed to Cell 3


In [ ]:
# Cell 3: HuggingFace login (needed for gated dataset ai4bharat/indicvoices_r)
from huggingface_hub import login
login()   # Paste your HF token when prompted

In [ ]:
# Cell 4: Download and load Oshara/xtts-v2-nepali, patch config to accept "hi"

import os, torch, json
from huggingface_hub import snapshot_download
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import Xtts

# ── Download epoch-10 checkpoint ──────────────────────────────────────────────
print("⬇️  Downloading Oshara/xtts-v2-nepali (epoch-10 only)…")
model_dir = snapshot_download(
    "Oshara/xtts-v2-nepali",
    allow_patterns=["epoch-10/*"],
    cache_dir="/content/oshara_xtts"
)
checkpoint_dir = os.path.join(model_dir, "epoch-10")
print(f"✅ Model at: {checkpoint_dir}")

# ── Patch config.json to register "hi" alongside "ne" ─────────────────────────
config_path = os.path.join(checkpoint_dir, "config.json")
with open(config_path, "r", encoding="utf-8") as f:
    cfg_data = json.load(f)

# The Oshara model exposes languages as a list in config; add "hi" if missing
if "languages" in cfg_data:
    langs = cfg_data["languages"]
    if "hi" not in langs:
        langs.append("hi")
        cfg_data["languages"] = langs
        with open(config_path, "w", encoding="utf-8") as f:
            json.dump(cfg_data, f, ensure_ascii=False, indent=2)
        print("✅ Patched config.json: added 'hi' to supported languages.")
    else:
        print("ℹ️  'hi' already in config languages.")
else:
    print("⚠️  No 'languages' key in config — skipping patch (model may accept all langs).")

# ── Load model ─────────────────────────────────────────────────────────────────
config = XttsConfig()
config.load_json(config_path)

model = Xtts.init_from_config(config)
model.load_checkpoint(
    config,
    checkpoint_path=os.path.join(checkpoint_dir, "model.pth"),
    vocab_path=os.path.join(checkpoint_dir, "vocab.json"),
    speaker_file_path=os.path.join(checkpoint_dir, "speakers_xtts.pth"),
    eval=True,
)
model.cuda()
print("✅ Oshara XTTS-v2 model loaded on GPU.")

⬇️  Downloading Oshara/xtts-v2-nepali (epoch-10 only)…


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

✅ Model at: /content/oshara_xtts/models--Oshara--xtts-v2-nepali/snapshots/1ef72e4a13e201409a895ef45c36b38adbe324d8/epoch-10
ℹ️  'hi' already in config languages.


[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 255), got 50256. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 255), got 50256. This may result in unexpected behavior.
[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 607), got 50256. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 607), got 50256. This may result in unexpected behavior.


✅ Oshara XTTS-v2 model loaded on GPU.


In [ ]:
# Cell 5 (FINAL): Stream IndicVoices_R Hindi — datasets 4.0.0 compatible
# audio column returns an AudioDecoder object, not a dict

import random
import numpy as np
from datasets import load_dataset

LANG_CONFIG = "Hindi"
SEED        = 42
COLLECT_MAX = 15
SCAN_LIMIT  = 4000
random.seed(SEED)

def extract_audio(row):
    """
    datasets 4.0.0: row["audio"] is an AudioDecoder object.
    datasets <4.0.0: row["audio"] is a dict {"array": ..., "sampling_rate": ...}.
    Handle both.
    """
    audio = row["audio"]
    if isinstance(audio, dict):
        arr = np.array(audio["array"], dtype=np.float32)
        sr  = int(audio["sampling_rate"])
    else:
        # AudioDecoder object — call get_all_samples()
        samples = audio.get_all_samples()
        arr = np.array(samples.data, dtype=np.float32)
        sr  = int(samples.sample_rate)
    # Flatten to mono if needed
    if arr.ndim > 1:
        arr = arr.mean(axis=1)
    return arr, sr

# ── Pass 1: scan for best male + female speaker ───────────────────────────────
print(f"🔄 Streaming IndicVoices_R ({LANG_CONFIG}) — train split…")
ds_stream = load_dataset(
    "ai4bharat/indicvoices_r",
    LANG_CONFIG,
    split="train",
    streaming=True,
)

speaker_buckets: dict = {}
print("🔍 Pass 1: scanning for speaker IDs by gender…")
for i, row in enumerate(ds_stream):
    if i >= SCAN_LIMIT:
        break
    sid    = row.get("speaker_id", "")
    gender = row.get("gender", "").strip().lower()
    if not sid or gender not in ("male", "female"):
        continue
    if sid not in speaker_buckets:
        speaker_buckets[sid] = {"gender": gender, "count": 0}
    speaker_buckets[sid]["count"] += 1

male_candidates = sorted(
    [(s, v["count"]) for s, v in speaker_buckets.items() if v["gender"] == "male"],
    key=lambda x: x[1], reverse=True
)
female_candidates = sorted(
    [(s, v["count"]) for s, v in speaker_buckets.items() if v["gender"] == "female"],
    key=lambda x: x[1], reverse=True
)

if not male_candidates or not female_candidates:
    raise ValueError(
        f"Could not find both genders in first {SCAN_LIMIT} rows. "
        f"Male: {len(male_candidates)}, Female: {len(female_candidates)}. "
        "Try increasing SCAN_LIMIT."
    )

chosen_male_id   = male_candidates[0][0]
chosen_female_id = female_candidates[0][0]
print(f"✅ Chosen male speaker   : {chosen_male_id}  ({male_candidates[0][1]} utts)")
print(f"✅ Chosen female speaker : {chosen_female_id} ({female_candidates[0][1]} utts)")

# ── Pass 2: collect audio + text ─────────────────────────────────────────────
print("🔄 Pass 2: collecting utterances…")
ds_stream2 = load_dataset(
    "ai4bharat/indicvoices_r",
    LANG_CONFIG,
    split="train",
    streaming=True,
)

male_utts   = []
female_utts = []

for row in ds_stream2:
    sid    = row.get("speaker_id", "")
    gender = row.get("gender", "").strip().lower()

    if sid == chosen_male_id and len(male_utts) < COLLECT_MAX:
        arr, sr = extract_audio(row)
        male_utts.append({"array": arr, "sr": sr, "text": row["text"]})

    elif sid == chosen_female_id and len(female_utts) < COLLECT_MAX:
        arr, sr = extract_audio(row)
        female_utts.append({"array": arr, "sr": sr, "text": row["text"]})

    if len(male_utts) >= COLLECT_MAX and len(female_utts) >= COLLECT_MAX:
        break

print(f"✅ Collected {len(male_utts)} male, {len(female_utts)} female utterances.")

if len(male_utts) < 5 or len(female_utts) < 5:
    raise ValueError(
        f"Not enough utterances: {len(male_utts)} male, {len(female_utts)} female. "
        "The chosen speakers may not appear densely enough in the stream. "
        "Try increasing SCAN_LIMIT or COLLECT_MAX."
    )

🔄 Streaming IndicVoices_R (Hindi) — train split…


Resolving data files:   0%|          | 0/246 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/99 [00:00<?, ?it/s]

🔍 Pass 1: scanning for speaker IDs by gender…
✅ Chosen male speaker   : S4259588400343717  (69 utts)
✅ Chosen female speaker : S4259569100343205 (63 utts)
🔄 Pass 2: collecting utterances…


Resolving data files:   0%|          | 0/246 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/99 [00:00<?, ?it/s]

✅ Collected 15 male, 15 female utterances.


In [ ]:
# Cell 6: Split into train (reference pool) and test (ground-truth), save wavs

import os, scipy.signal, soundfile as sf
import numpy as np

OUT_DIR = "/content/xtts_hindi_eval"
os.makedirs(f"{OUT_DIR}/reference", exist_ok=True)
os.makedirs(f"{OUT_DIR}/ground_truth/male",   exist_ok=True)
os.makedirs(f"{OUT_DIR}/ground_truth/female", exist_ok=True)
os.makedirs(f"{OUT_DIR}/generated/eval_set/male",   exist_ok=True)
os.makedirs(f"{OUT_DIR}/generated/eval_set/female", exist_ok=True)

TARGET_SR = 22050   # XTTS-v2 expects 22050 Hz reference audio

def resample_to(arr: np.ndarray, src_sr: int, tgt_sr: int) -> np.ndarray:
    if src_sr == tgt_sr:
        return arr
    n_samples = int(len(arr) * tgt_sr / src_sr)
    return scipy.signal.resample(arr, n_samples).astype(np.float32)

def save_wav(arr: np.ndarray, sr: int, path: str):
    # Normalize to avoid clipping
    peak = np.abs(arr).max()
    if peak > 0:
        arr = arr / peak * 0.95
    sf.write(path, arr, sr)

# ── Helper: split utterances ──────────────────────────────────────────────────
def split_train_test(utts, n_train=10, n_test=5):
    """First n_train utts = reference pool; last n_test = unseen ground truth."""
    random.shuffle(utts)
    return utts[:n_train], utts[n_train:n_train + n_test]

male_train,   male_test   = split_train_test(male_utts,   n_train=10, n_test=5)
female_train, female_test = split_train_test(female_utts, n_train=10, n_test=5)

# ── Save ground-truth test wavs (unseen, used to compare with generated) ──────
for i, utt in enumerate(male_test):
    arr = resample_to(utt["array"], utt["sr"], TARGET_SR)
    save_wav(arr, TARGET_SR, f"{OUT_DIR}/ground_truth/male/gt_{i+1:02d}.wav")

for i, utt in enumerate(female_test):
    arr = resample_to(utt["array"], utt["sr"], TARGET_SR)
    save_wav(arr, TARGET_SR, f"{OUT_DIR}/ground_truth/female/gt_{i+1:02d}.wav")

# ── Concatenate train utterances → single reference WAV per gender ────────────
def build_reference_wav(utts, out_path, target_sr=TARGET_SR):
    """Concatenate multiple utterances with 0.3 s silence → reference WAV."""
    silence = np.zeros(int(target_sr * 0.3), dtype=np.float32)
    parts = []
    for utt in utts:
        arr = resample_to(utt["array"], utt["sr"], target_sr)
        parts.append(arr)
        parts.append(silence)
    combined = np.concatenate(parts)
    # Keep max 30 s (XTTS uses up to 30 s of reference)
    max_samples = target_sr * 30
    combined = combined[:max_samples]
    save_wav(combined, target_sr, out_path)
    duration = len(combined) / target_sr
    print(f"  Reference WAV: {out_path}  ({duration:.1f} s)")

male_ref_path   = f"{OUT_DIR}/reference/male_reference.wav"
female_ref_path = f"{OUT_DIR}/reference/female_reference.wav"

print("Building reference WAVs…")
build_reference_wav(male_train,   male_ref_path)
build_reference_wav(female_train, female_ref_path)

# ── Save metadata ─────────────────────────────────────────────────────────────
import json

meta = {
    "male_speaker_id":   chosen_male_id,
    "female_speaker_id": chosen_female_id,
    "male_train_texts":  [u["text"] for u in male_train],
    "female_train_texts":[u["text"] for u in female_train],
    "male_test_texts":   [u["text"] for u in male_test],
    "female_test_texts": [u["text"] for u in female_test],
}
with open(f"{OUT_DIR}/speaker_metadata.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print("\n✅ Train/test split complete.")
print(f"   Male   → {len(male_train)} train (reference) | {len(male_test)} test (ground truth)")
print(f"   Female → {len(female_train)} train (reference) | {len(female_test)} test (ground truth)")
print(f"   Reference WAVs saved to: {OUT_DIR}/reference/")
print(f"   Ground truth WAVs saved to: {OUT_DIR}/ground_truth/")

Building reference WAVs…
  Reference WAV: /content/xtts_hindi_eval/reference/male_reference.wav  (30.0 s)
  Reference WAV: /content/xtts_hindi_eval/reference/female_reference.wav  (30.0 s)

✅ Train/test split complete.
   Male   → 10 train (reference) | 5 test (ground truth)
   Female → 10 train (reference) | 5 test (ground truth)
   Reference WAVs saved to: /content/xtts_hindi_eval/reference/
   Ground truth WAVs saved to: /content/xtts_hindi_eval/ground_truth/


In [ ]:
# Cell 7 (FINAL): Generate 20 eval sentences — loads reference WAV via
# soundfile directly, bypassing torchaudio entirely to avoid stub conflicts.

# Add these 4 lines at the TOP of Cell 7, before everything else
import torch, soundfile as _sf4, numpy as _np4, scipy.signal as _sig4

def _load_audio_final(audiopath, sampling_rate):
    data, sr = _sf4.read(str(audiopath), dtype="float32", always_2d=False)
    if data.ndim > 1:
        data = data.mean(axis=1)
    if sr != sampling_rate:
        n    = int(len(data) * sampling_rate / sr)
        data = _sig4.resample(data, n).astype(_np4.float32)
    data = _np4.clip(data, -1.0, 1.0)
    return torch.from_numpy(data).unsqueeze(0)

# Patch into all model method globals
import types
for attr_name in dir(model):
    try:
        attr = getattr(model, attr_name)
        if isinstance(attr, types.MethodType):
            g = attr.__func__.__globals__
            if "load_audio" in g or "xtts" in str(g.get("__file__", "")):
                g["load_audio"] = _load_audio_final
    except Exception:
        pass
model.get_conditioning_latents.__func__.__globals__["load_audio"] = _load_audio_final
model._clone_voice.__func__.__globals__["load_audio"] = _load_audio_final
print("✅ load_audio patched")


import os, json, torch, numpy as np, soundfile as sf, scipy.signal, tempfile

# ── Auto-reload model if not in scope ────────────────────────────────────────
def _load_oshara_model():
    import os, json
    from huggingface_hub import snapshot_download
    from TTS.tts.configs.xtts_config import XttsConfig
    from TTS.tts.models.xtts import Xtts

    print("🔄 Loading Oshara XTTS-v2 model…")
    model_dir = snapshot_download(
        "Oshara/xtts-v2-nepali",
        allow_patterns=["epoch-10/*"],
        cache_dir="/content/oshara_xtts"
    )
    checkpoint_dir = os.path.join(model_dir, "epoch-10")
    config_path    = os.path.join(checkpoint_dir, "config.json")

    with open(config_path, "r", encoding="utf-8") as f:
        cfg_data = json.load(f)
    if "languages" in cfg_data and "hi" not in cfg_data["languages"]:
        cfg_data["languages"].append("hi")
        with open(config_path, "w", encoding="utf-8") as f:
            json.dump(cfg_data, f, ensure_ascii=False, indent=2)

    config = XttsConfig()
    config.load_json(config_path)
    mdl = Xtts.init_from_config(config)
    mdl.load_checkpoint(
        config,
        checkpoint_path=os.path.join(checkpoint_dir, "model.pth"),
        vocab_path=os.path.join(checkpoint_dir, "vocab.json"),
        speaker_file_path=os.path.join(checkpoint_dir, "speakers_xtts.pth"),
        eval=True,
    )
    mdl.cuda()
    print("✅ Model loaded on GPU.")
    return mdl, config

try:
    model
    config
    print("✅ model already in scope")
except NameError:
    model, config = _load_oshara_model()

# ── Paths ─────────────────────────────────────────────────────────────────────
OUT_DIR         = "/content/xtts_hindi_eval"
male_ref_path   = f"{OUT_DIR}/reference/male_reference.wav"
female_ref_path = f"{OUT_DIR}/reference/female_reference.wav"

for p in [male_ref_path, female_ref_path]:
    if not os.path.exists(p):
        raise FileNotFoundError(f"Reference WAV missing: {p} — run Cells 5 & 6 first.")
print("✅ Reference WAVs found")

# ── Helper: load WAV via soundfile → resample to 22050 → temp file ────────────
# XTTS synthesize() accepts a file path for speaker_wav. It internally calls
# torchaudio.load() on that path. To avoid our AudioDecoder stub interfering,
# we pre-load with soundfile, resample to 22050 Hz, write a clean temp WAV,
# and pass that path. torchaudio reads the temp WAV natively (no stub involved).

def prepare_ref_wav(path: str, target_sr: int = 22050) -> str:
    """
    Load path with soundfile, resample to target_sr, write to a temp WAV.
    Returns the temp file path.
    """
    data, sr = sf.read(path, dtype="float32", always_2d=False)
    if data.ndim > 1:
        data = data.mean(axis=1)
    if sr != target_sr:
        n    = int(len(data) * target_sr / sr)
        data = scipy.signal.resample(data, n).astype(np.float32)
    # Normalize
    peak = np.abs(data).max()
    if peak > 0:
        data = data / peak * 0.95
    # Write to a real WAV file that torchaudio can open natively
    tmp = tempfile.NamedTemporaryFile(suffix=".wav", delete=False)
    sf.write(tmp.name, data, target_sr)
    tmp.close()
    return tmp.name

print("🔧 Preparing reference WAVs (resample → temp files)…")
male_ref_tmp   = prepare_ref_wav(male_ref_path)
female_ref_tmp = prepare_ref_wav(female_ref_path)
print(f"   Male   temp ref: {male_ref_tmp}")
print(f"   Female temp ref: {female_ref_tmp}")

# ── Eval set ──────────────────────────────────────────────────────────────────
EVAL_JSON = {
    "vowels_and_consonants":      {"id": "HIN_01", "text": "एक आम आदमी और औरत इमली के पेड़ के नीचे बैठकर ऊन बुन रहे हैं।"},
    "velars_gutturals":           {"id": "HIN_02", "text": "कमल और काव्या ने खेत से कद्दू उखाड़ा, फिर गरम घी का घड़ा उठाकर घर की ओर भागे।"},
    "retroflexes":                {"id": "HIN_03", "text": "षट्कोण के अंदर रखे ढक्कन और डमरू को देखकर ठग टमाटर टोकरी में डालकर डर गया।"},
    "palatals_and_nasals":        {"id": "HIN_04", "text": "चंचल छतरी लेकर झमाझम बारिश में जंगल की ओर चली गई।"},
    "labials_and_aspirated":      {"id": "HIN_05", "text": "भालू ने भारी पेड़ पर बैठकर मीठा फल खाया और पानी पी लिया।"},
    "loan_words_nukta":           {"id": "HIN_06", "text": "ज़रा फ़िक्र मत करो, क़लम से ख़त लिखकर ग़ज़ल का मज़ा लो।"},
    "complex_conjuncts":          {"id": "HIN_07", "text": "ज्ञानी ऋषि और श्रमिक ने क्षमा, विज्ञान और त्रिशूल का महत्व समझाया।"},
    "dentals_and_visarga":        {"id": "HIN_08", "text": "दुःख मत कर, स्वतः ही नया धन प्राप्त होगा और धर्म की जीत होगी।"},
    "flaps_and_chandrabindu":     {"id": "HIN_09", "text": "गाँव में पाँच ऊँट, एक साँड़, और बड़ी सूँड वाले बूढ़े हाथी खड़े थे।"},
    "approximants_and_sibilants": {"id": "HIN_10", "text": "यश और श्वेता बहुत विश्वास के साथ विद्यालय में योग और व्यायाम सीखने गए।"},
    "english_loan_words":         {"id": "HIN_11", "text": "आजकल के डॉक्टर और इंजीनियर मॉडर्न स्कूल के प्रोजेक्ट पर काम कर रहे हैं।"},
    "heavy_geminates":            {"id": "HIN_12", "text": "बिल्ली ने चम्मच से मक्खन चाटा और छप्पर पर कूदकर गुब्बारा फोड़ दिया।"},
    "ha_placement":               {"id": "HIN_13", "text": "हम कल सुबह शहर के उस बड़े महल की वजह से वहाँ ठहरेंगे।"},
    "vowel_hiatus":               {"id": "HIN_14", "text": "भैया, कौआ उड़ गया, अब आइए और मुझे बताइए कि मैं वहाँ कैसे जाऊँगा?"},
    "sanskrit_tatsama":           {"id": "HIN_15", "text": "इस उज्ज्वल और महत्त्वपूर्ण कार्य के लिए प्राचीन संस्कृति और प्रौद्योगिकी का ज्ञान अनिवार्य है।"},
    "prosody_and_punctuation":    {"id": "HIN_16", "text": "वाह! तुमने तो कमाल कर दिया; लेकिन, क्या तुम्हें सच में लगता है कि यह मुमकिन है?"},
    "perso_arabic_nukta":         {"id": "HIN_17", "text": "ख़ौफ़नाक तूफ़ान के बाज़ू में खड़े फ़क़ीर ने क़र्ज़ माफ़ करने की गुज़ारिश की।"},
    "number_normalization":       {"id": "HIN_18", "text": "सेठ जी ने पचहत्तर प्रतिशत मुनाफ़े के साथ कुल चौवन हज़ार रुपये नकद कमाए।"},
    "consonant_clusters_r":       {"id": "HIN_19", "text": "ट्रेन स्टेशन से प्रस्थान कर चुकी है, कृपया अपने ट्रक को क्रॉसिंग से दूर रखें।"},
    "alliteration_rapid":         {"id": "HIN_20", "text": "चंदू के चाचा ने चाँदी के चम्मच से चटपटी चटनी चखाई और चंपारण चले गए।"},
}

GEN_MALE   = f"{OUT_DIR}/generated/eval_set/male"
GEN_FEMALE = f"{OUT_DIR}/generated/eval_set/female"
os.makedirs(GEN_MALE,   exist_ok=True)
os.makedirs(GEN_FEMALE, exist_ok=True)

LANGUAGE    = "hi"
SAMPLE_RATE = 24000

SYNTH_KWARGS = dict(
    language=LANGUAGE,
    temperature=0.65,
    repetition_penalty=5.0,
    top_k=50,
    top_p=0.85,
    speed=1.0,
    enable_text_splitting=True,
)

def synthesize_sentence(text: str, ref_wav_path: str, out_path: str):
    out = model.synthesize(text, config, speaker_wav=ref_wav_path, **SYNTH_KWARGS)
    wav = np.array(out["wav"], dtype=np.float32)
    peak = np.abs(wav).max()
    if peak > 0:
        wav = wav / peak * 0.95
    sf.write(out_path, wav, SAMPLE_RATE)

# ── Generate ──────────────────────────────────────────────────────────────────
results_log = []

for gender, ref_tmp, gen_dir in [
    ("male",   male_ref_tmp,   GEN_MALE),
    ("female", female_ref_tmp, GEN_FEMALE),
]:
    print(f"\n🎙️  Generating {gender} voice — 20 sentences…")
    for category, item in EVAL_JSON.items():
        sent_id  = item["id"]
        text     = item["text"]
        out_path = os.path.join(gen_dir, f"{sent_id}_{gender}.wav")
        print(f"  [{sent_id}] {category[:30]:30s}", end="  ")
        try:
            synthesize_sentence(text, ref_tmp, out_path)
            print(f"✅  → {os.path.basename(out_path)}")
            results_log.append({"id": sent_id, "gender": gender,
                                 "category": category, "status": "ok",
                                 "path": out_path})
        except Exception as e:
            print(f"❌  ERROR: {e}")
            results_log.append({"id": sent_id, "gender": gender,
                                 "category": category,
                                 "status": f"error: {e}", "path": None})

# Cleanup temp files
import os as _os
for p in [male_ref_tmp, female_ref_tmp]:
    try: _os.unlink(p)
    except: pass

with open(f"{OUT_DIR}/generation_log.json", "w", encoding="utf-8") as f:
    json.dump(results_log, f, ensure_ascii=False, indent=2)

ok_count = sum(1 for r in results_log if r["status"] == "ok")
print(f"\n✅ Generation complete: {ok_count}/{len(results_log)} sentences succeeded.")
print(f"   Output: {OUT_DIR}/generated/eval_set/")

✅ load_audio patched
✅ model already in scope
✅ Reference WAVs found
🔧 Preparing reference WAVs (resample → temp files)…
   Male   temp ref: /tmp/tmp69kr1jfz.wav
   Female temp ref: /tmp/tmpslofgjoo.wav

🎙️  Generating male voice — 20 sentences…
  [HIN_01] vowels_and_consonants           ✅  → HIN_01_male.wav
  [HIN_02] velars_gutturals                ✅  → HIN_02_male.wav
  [HIN_03] retroflexes                     ✅  → HIN_03_male.wav
  [HIN_04] palatals_and_nasals             ✅  → HIN_04_male.wav
  [HIN_05] labials_and_aspirated           ✅  → HIN_05_male.wav
  [HIN_06] loan_words_nukta                ✅  → HIN_06_male.wav
  [HIN_07] complex_conjuncts               ✅  → HIN_07_male.wav
  [HIN_08] dentals_and_visarga             ✅  → HIN_08_male.wav
  [HIN_09] flaps_and_chandrabindu          ✅  → HIN_09_male.wav
  [HIN_10] approximants_and_sibilants      ✅  → HIN_10_male.wav
  [HIN_11] english_loan_words              ✅  → HIN_11_male.wav
  [HIN_12] heavy_geminates                 ✅  → HI

In [ ]:
# Cell 8: Generate TTS for the 5 unseen test sentences per gender
# These can be compared directly against the saved ground-truth WAVs.

import json, os
import numpy as np
import soundfile as sf

OUT_DIR    = "/content/xtts_hindi_eval"
GEN_TEST_M = f"{OUT_DIR}/generated/test_set/male"
GEN_TEST_F = f"{OUT_DIR}/generated/test_set/female"
os.makedirs(GEN_TEST_M, exist_ok=True)
os.makedirs(GEN_TEST_F, exist_ok=True)

SAMPLE_RATE = 24000

test_pairs = [
    ("male",   male_test,   male_ref_path,   GEN_TEST_M),
    ("female", female_test, female_ref_path, GEN_TEST_F),
]

test_log = []

for gender, test_utts, ref_wav, gen_dir in test_pairs:
    print(f"\n🎙️  Generating test-set ({gender}) — {len(test_utts)} sentences…")
    for i, utt in enumerate(test_utts):
        text     = utt["text"]
        out_path = os.path.join(gen_dir, f"test_{i+1:02d}_{gender}_gen.wav")
        gt_path  = f"{OUT_DIR}/ground_truth/{gender}/gt_{i+1:02d}.wav"

        print(f"  [{i+1}] {text[:55]}…", end="  ")
        try:
            out = model.synthesize(text, config, speaker_wav=ref_wav, **SYNTH_KWARGS)
            wav = np.array(out["wav"], dtype=np.float32)
            peak = np.abs(wav).max()
            if peak > 0:
                wav = wav / peak * 0.95
            sf.write(out_path, wav, SAMPLE_RATE)
            print(f"✅")
            test_log.append({"index": i+1, "gender": gender, "text": text,
                              "generated": out_path, "ground_truth": gt_path, "status": "ok"})
        except Exception as e:
            print(f"❌ {e}")
            test_log.append({"index": i+1, "gender": gender, "text": text,
                              "generated": None, "ground_truth": gt_path, "status": f"error: {e}"})

with open(f"{OUT_DIR}/test_generation_log.json", "w", encoding="utf-8") as f:
    json.dump(test_log, f, ensure_ascii=False, indent=2)

ok = sum(1 for r in test_log if r["status"] == "ok")
print(f"\n✅ Test-set generation: {ok}/{len(test_log)} succeeded.")


🎙️  Generating test-set (male) — 5 sentences…
  [1] जिसमें हमारे गाँव के बहुत सारे गरीब बच्चे पढ़ने के लिए …  ✅
  [2] अगर से मैं बात करूँ तो सभी धार्मिक स्थलों पर समय समय पर…  ✅
  [3] जी हाँ अगर से हम बात करें तो हमारे क्षेत्र में बहुत सार…  ✅
  [4] जैसे कि हम एक दो स्कूलों के बारे में आपको बताना चाहे तो…  ✅
  [5] जिसके वजह से सामान्य वर्ग के लोग अपने बच्चों को चाह कर …  ✅

🎙️  Generating test-set (female) — 5 sentences…
  [1] दिन पे दिन जो भ्रष्टाचार है वो हम लोगों को सामना करना प…  ✅
  [2] उन्हें बहुत सारे पैसा मिलता है यूट्यूब और इंस्टाग्राम क…  ✅
  [3] तो ऐसे ऐसे बहुत सारे सीन है जो मतलब और इंस्टाग्राम पर अ…  ✅
  [4] मैंने जो अपने कला की प्रतियोगिता मतलब जो प्रतियोगिता आत…  ✅
  [5] यू यूट्यूब पर कर रहे हैं तो ऐसे में उन्हें बेनिफिट मिला…  ✅

✅ Test-set generation: 10/10 succeeded.


In [ ]:
# Cell 9: Print final directory structure summary

import os

ROOT = "/content/xtts_hindi_eval"
for dirpath, dirnames, filenames in os.walk(ROOT):
    dirnames.sort()
    level = dirpath.replace(ROOT, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(dirpath)}/")
    sub = "  " * (level + 1)
    for f in sorted(filenames):
        size_kb = os.path.getsize(os.path.join(dirpath, f)) / 1024
        print(f"{sub}{f}  ({size_kb:.0f} KB)")

xtts_hindi_eval/
  generation_log.json  (7 KB)
  speaker_metadata.json  (11 KB)
  test_generation_log.json  (6 KB)
  wer_cer_results.json  (27 KB)
  generated/
    eval_set/
      female/
        HIN_01_female.wav  (237 KB)
        HIN_02_female.wav  (309 KB)
        HIN_03_female.wav  (285 KB)
        HIN_04_female.wav  (218 KB)
        HIN_05_female.wav  (194 KB)
        HIN_06_female.wav  (229 KB)
        HIN_07_female.wav  (272 KB)
        HIN_08_female.wav  (246 KB)
        HIN_09_female.wav  (218 KB)
        HIN_10_female.wav  (257 KB)
        HIN_11_female.wav  (255 KB)
        HIN_12_female.wav  (211 KB)
        HIN_13_female.wav  (222 KB)
        HIN_14_female.wav  (222 KB)
        HIN_15_female.wav  (298 KB)
        HIN_16_female.wav  (246 KB)
        HIN_17_female.wav  (242 KB)
        HIN_18_female.wav  (263 KB)
        HIN_19_female.wav  (259 KB)
        HIN_20_female.wav  (250 KB)
      male/
        HIN_01_male.wav  (229 KB)
        HIN_02_male.wav  (266 KB)
        HIN_

In [ ]:
# Cell 10: Zip the entire output folder and download it

import shutil
from google.colab import files

zip_path = "/content/xtts_hindi_eval_output"
print("📦 Zipping output folder…")
shutil.make_archive(zip_path, "zip", "/content/xtts_hindi_eval")
print(f"✅ Archive ready: {zip_path}.zip")
print("⬇️  Starting download…")
files.download(f"{zip_path}.zip")

📦 Zipping output folder…
✅ Archive ready: /content/xtts_hindi_eval_output.zip
⬇️  Starting download…


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Cell 11: Install ASR evaluation dependencies
import subprocess, sys

def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

pip("openai-whisper", "jiwer")
print("✅ whisper + jiwer installed")

✅ whisper + jiwer installed


In [ ]:
# Cell 12: Compute WER and CER for all generated WAVs using Whisper medium
# Compares:
#   (A) eval_set : generated speech vs. original reference text (HIN_01–HIN_20)
#   (B) test_set : generated speech vs. ground-truth transcription from dataset

import os, json, whisper, torch
from jiwer import wer, cer

OUT_DIR = "/content/xtts_hindi_eval"

# ── Load Whisper medium ───────────────────────────────────────────────────────
print("🔄 Loading Whisper medium…")
asr = whisper.load_model("medium", device="cuda" if torch.cuda.is_available() else "cpu")
print("✅ Whisper medium loaded")

WHISPER_OPTS = dict(language="hi", task="transcribe", fp16=torch.cuda.is_available())

def transcribe(wav_path: str) -> str:
    """Transcribe a WAV file with Whisper, return normalised Hindi text."""
    result = asr.transcribe(wav_path, **WHISPER_OPTS)
    return result["text"].strip()

def normalise(text: str) -> str:
    """
    Minimal normalisation for Hindi WER/CER:
    - strip leading/trailing whitespace
    - collapse multiple spaces
    - remove punctuation that Whisper sometimes adds/omits inconsistently
    """
    import re
    text = text.strip()
    text = re.sub(r"[।॥,.!?;:\"'()\\-]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# ── Eval-set scoring (generated vs. reference text) ──────────────────────────
print("\n📊 Scoring eval set (HIN_01–HIN_20)…")

EVAL_TEXTS = {
    "HIN_01": "एक आम आदमी और औरत इमली के पेड़ के नीचे बैठकर ऊन बुन रहे हैं।",
    "HIN_02": "कमल और काव्या ने खेत से कद्दू उखाड़ा, फिर गरम घी का घड़ा उठाकर घर की ओर भागे।",
    "HIN_03": "षट्कोण के अंदर रखे ढक्कन और डमरू को देखकर ठग टमाटर टोकरी में डालकर डर गया।",
    "HIN_04": "चंचल छतरी लेकर झमाझम बारिश में जंगल की ओर चली गई।",
    "HIN_05": "भालू ने भारी पेड़ पर बैठकर मीठा फल खाया और पानी पी लिया।",
    "HIN_06": "ज़रा फ़िक्र मत करो, क़लम से ख़त लिखकर ग़ज़ल का मज़ा लो।",
    "HIN_07": "ज्ञानी ऋषि और श्रमिक ने क्षमा, विज्ञान और त्रिशूल का महत्व समझाया।",
    "HIN_08": "दुःख मत कर, स्वतः ही नया धन प्राप्त होगा और धर्म की जीत होगी।",
    "HIN_09": "गाँव में पाँच ऊँट, एक साँड़, और बड़ी सूँड वाले बूढ़े हाथी खड़े थे।",
    "HIN_10": "यश और श्वेता बहुत विश्वास के साथ विद्यालय में योग और व्यायाम सीखने गए।",
    "HIN_11": "आजकल के डॉक्टर और इंजीनियर मॉडर्न स्कूल के प्रोजेक्ट पर काम कर रहे हैं।",
    "HIN_12": "बिल्ली ने चम्मच से मक्खन चाटा और छप्पर पर कूदकर गुब्बारा फोड़ दिया।",
    "HIN_13": "हम कल सुबह शहर के उस बड़े महल की वजह से वहाँ ठहरेंगे।",
    "HIN_14": "भैया, कौआ उड़ गया, अब आइए और मुझे बताइए कि मैं वहाँ कैसे जाऊँगा?",
    "HIN_15": "इस उज्ज्वल और महत्त्वपूर्ण कार्य के लिए प्राचीन संस्कृति और प्रौद्योगिकी का ज्ञान अनिवार्य है।",
    "HIN_16": "वाह! तुमने तो कमाल कर दिया; लेकिन, क्या तुम्हें सच में लगता है कि यह मुमकिन है?",
    "HIN_17": "ख़ौफ़नाक तूफ़ान के बाज़ू में खड़े फ़क़ीर ने क़र्ज़ माफ़ करने की गुज़ारिश की।",
    "HIN_18": "सेठ जी ने पचहत्तर प्रतिशत मुनाफ़े के साथ कुल चौवन हज़ार रुपये नकद कमाए।",
    "HIN_19": "ट्रेन स्टेशन से प्रस्थान कर चुकी है, कृपया अपने ट्रक को क्रॉसिंग से दूर रखें।",
    "HIN_20": "चंदू के चाचा ने चाँदी के चम्मच से चटपटी चटनी चखाई और चंपारण चले गए।",
}

eval_results = []

for gender in ["male", "female"]:
    gen_dir = f"{OUT_DIR}/generated/eval_set/{gender}"
    print(f"\n  Gender: {gender}")
    print(f"  {'ID':<8} {'WER':>6} {'CER':>6}  Hypothesis")
    print(f"  {'-'*7} {'-'*6} {'-'*6}  {'-'*50}")

    for sent_id, ref_text in EVAL_TEXTS.items():
        wav_path = os.path.join(gen_dir, f"{sent_id}_{gender}.wav")
        if not os.path.exists(wav_path):
            print(f"  {sent_id:<8} {'MISSING':>6}")
            continue
        try:
            hyp  = transcribe(wav_path)
            ref_n = normalise(ref_text)
            hyp_n = normalise(hyp)
            w = wer(ref_n, hyp_n)
            c = cer(ref_n, hyp_n)
            print(f"  {sent_id:<8} {w:>6.3f} {c:>6.3f}  {hyp[:60]}")
            eval_results.append({
                "gender": gender, "id": sent_id,
                "reference": ref_text, "hypothesis": hyp,
                "wer": round(w, 4), "cer": round(c, 4),
                "set": "eval"
            })
        except Exception as e:
            print(f"  {sent_id:<8} ERROR: {e}")

# ── Test-set scoring (generated vs. ground-truth transcription) ───────────────
print("\n📊 Scoring test set (unseen utterances)…")

meta_path = f"{OUT_DIR}/speaker_metadata.json"
with open(meta_path, "r", encoding="utf-8") as f:
    meta = json.load(f)

test_results = []

for gender in ["male", "female"]:
    texts_key = f"{gender}_test_texts"
    test_texts = meta.get(texts_key, [])
    gen_dir    = f"{OUT_DIR}/generated/test_set/{gender}"
    print(f"\n  Gender: {gender}")
    print(f"  {'#':<5} {'WER':>6} {'CER':>6}  Hypothesis")
    print(f"  {'-'*4} {'-'*6} {'-'*6}  {'-'*50}")

    for i, ref_text in enumerate(test_texts, start=1):
        wav_path = os.path.join(gen_dir, f"test_{i:02d}_{gender}_gen.wav")
        if not os.path.exists(wav_path):
            print(f"  {i:<5} MISSING")
            continue
        try:
            hyp   = transcribe(wav_path)
            ref_n  = normalise(ref_text)
            hyp_n  = normalise(hyp)
            w = wer(ref_n, hyp_n)
            c = cer(ref_n, hyp_n)
            print(f"  {i:<5} {w:>6.3f} {c:>6.3f}  {hyp[:60]}")
            test_results.append({
                "gender": gender, "index": i,
                "reference": ref_text, "hypothesis": hyp,
                "wer": round(w, 4), "cer": round(c, 4),
                "set": "test"
            })
        except Exception as e:
            print(f"  {i:<5} ERROR: {e}")

# ── Aggregate summary ─────────────────────────────────────────────────────────
all_results = eval_results + test_results

print("\n" + "="*60)
print("AGGREGATE SUMMARY")
print("="*60)

for subset in ["eval", "test"]:
    for gender in ["male", "female"]:
        rows = [r for r in all_results if r["set"] == subset and r["gender"] == gender]
        if not rows:
            continue
        avg_wer = sum(r["wer"] for r in rows) / len(rows)
        avg_cer = sum(r["cer"] for r in rows) / len(rows)
        print(f"  {subset:5s} | {gender:6s} | avg WER: {avg_wer:.3f} | avg CER: {avg_cer:.3f} | n={len(rows)}")

print("="*60)

# ── Save full results ─────────────────────────────────────────────────────────
results_path = f"{OUT_DIR}/wer_cer_results.json"
with open(results_path, "w", encoding="utf-8") as f:
    json.dump(all_results, f, ensure_ascii=False, indent=2)
print(f"\n✅ Full results saved → {results_path}")

🔄 Loading Whisper medium…


100%|█████████████████████████████████████| 1.42G/1.42G [00:22<00:00, 68.2MiB/s]


✅ Whisper medium loaded

📊 Scoring eval set (HIN_01–HIN_20)…

  Gender: male
  ID          WER    CER  Hypothesis
  ------- ------ ------  --------------------------------------------------
  HIN_01    0.267  0.136  एक आम आदमी और औरत इमली के पेर के निजे बविटकर उन बुन रहे हैं।
  HIN_02    0.556  0.200  कमल और काब्याने खेछ से कदू खारा, फिर गरम गीका घरा उठाकर घर क
  HIN_03    0.438  0.247  सथकॉन के अंदर रखे धाखन और दमृ को देखकर थाग तमाटा तोकरी में ड
  HIN_04    0.455  0.167  चंचेल छतरी लेकर जमा जम भारिश में जंगल की उर्थ चली गई।
  HIN_05    0.615  0.255  भालूने भारी पेर पर बव़िटकर मिद्हा फर खाया और पानी पिलिया।
  HIN_06    0.750  0.170  जरा फिक्र मत करो, कलम से खात लिख कर गजल का मजालो।
  HIN_07    0.667  0.266  ग्यानी, रिशी और स्रमितने छमा, भिज्यान और त्रेसूल का महत्व सम
  HIN_08    0.286  0.119  दुका मत कर, सुतही नया धन प्राप्त होगा और धर्म की जीथ होगी.
  HIN_09    0.643  0.302  गाउ में पाचुट, एक सार्ड और बरी सुढ भाले बूरे हाति खड़े थे।
  HIN_10    0.571  0.232  यश और स्वेता बहुत भी स्वास